# DSI tract

Port of `packages/niivue/examples/tract.dsi.html`. Loads a background volume and a DSI-Studio TinyTrack file, with controls for color, radius, decimation, and X-ray blending.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

from ipyniivue import NiiVue

BASE_URL = "https://niivue.github.io/mono"
VOLUMES = f"{BASE_URL}/volumes"
MESHES = f"{BASE_URL}/meshes"

nv = NiiVue(background_color=[0, 0, 0, 1], slice_type=4, backend="webgl2")

slice_type = widgets.Dropdown(options=[("Render", 4), ("A+C+S+R", 3)], value=4, description="Slice")
fiber_color = widgets.Dropdown(
    options=[("Local direction", ""), ("Global direction", "global"), ("Fixed", "fixed")],
    value="",
    description="Color",
)
radius = widgets.IntSlider(value=5, min=0, max=20, step=1, description="Radius")
decimation = widgets.IntSlider(value=1, min=1, max=20, step=1, description="Decimation")
xray = widgets.Checkbox(value=False, description="XRay")


def update_slice(change):
    nv.slice_type = change["new"]


def update_radius(change):
    nv.set_tract_options(0, {"fiberRadius": change["new"] * 0.1})


def update_decimation(change):
    nv.set_tract_options(0, {"decimation": change["new"]})


def update_color(change):
    nv.set_tract_options(0, {"colorBy": change["new"]})


def update_xray(change):
    nv.mesh_x_ray = 0.05 if change["new"] else 0.0


slice_type.observe(update_slice, names="value")
radius.observe(update_radius, names="value")
decimation.observe(update_decimation, names="value")
fiber_color.observe(update_color, names="value")
xray.observe(update_xray, names="value")

controls = widgets.VBox(
    [
        widgets.HBox([slice_type, fiber_color, xray]),
        widgets.HBox([radius, decimation]),
    ]
)
display(widgets.VBox([controls, nv]))

nv.clip_plane_color = [0, 0, 0, 0]
nv.set_clip_planes([[0.2, 180, 80], [-0.15, 0, -80]])
nv.load_volumes([{"url": f"{VOLUMES}/mni152.nii.gz"}])
nv.load_meshes(
    [
        {
            "url": f"{MESHES}/TR_S_R.tt.gz",
            "rgba255": [0, 255, 255, 255],
            "tractOptions": {"fiberRadius": 0.5},
        }
    ]
)